# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect, and verify every column name before using it

Nothing here is part of the contract yet. This section only makes sure I am pointed at the right
release and that the columns I am about to write sentences about actually exist, under the names
I think they have.

In [5]:
%pip -q install duckdb

# Token: env var -> Colab Secret -> prompt (last resort).
# NEVER paste a token into a cell -- this repo is public. Colab: use the key panel (Secrets)
# and name the secret HF_TOKEN.
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [6]:
import duckdb, pandas as pd, numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL   = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                       # mid-panel development month (NOT the final month)

T = {
    "dim_clients":   f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":   f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_month":    f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
    "fact_sample":   f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query90":  f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# COUNT(*) over Parquet reads metadata, not data -- near-free, and it proves I am pointed at the
# build the docs describe. Published: 104 / 519,606 / 78,835,655 / 2,414,248.
published = {"dim_clients": 104, "dim_content": 519_606,
             "fact_daily": 78_835_655, "fact_query90": 2_414_248}

for name in ["dim_clients", "dim_content", "fact_daily", "fact_query90"]:
    n = con.sql(f"SELECT COUNT(*) FROM {T[name]}").fetchone()[0]
    flag = "matches docs" if published[name] == n else f"DIFFERS from docs ({published[name]:,})"
    print(f"{name:14} {n:>12,} rows   <- {flag}")

dim_clients             104 rows   <- matches docs
dim_content         519,606 rows   <- matches docs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily       78,835,655 rows   <- matches docs
fact_query90      2,414,248 rows   <- matches docs


In [7]:
# --- Column-name handshake -------------------------------------------------
# Haris's rule: doubt a column name until you've verified it. I resolve every name I need
# against the live schema instead of trusting my memory or a doc.

def columns_of(relation):
    return list(con.sql(f"SELECT * FROM {relation} LIMIT 0").df().columns)

FACT_COLS   = columns_of(T["fact_month"])
CLIENT_COLS = columns_of(T["dim_clients"])

def resolve(candidates, available, label):
    for c in candidates:
        if c in available:
            return c
    raise KeyError(
        f"None of {candidates} found for '{label}'. Available columns:\n  " + ", ".join(available)
    )

COL = {
    "date":     resolve(["report_date", "date"],                                   FACT_COLS, "report date"),
    "client":   resolve(["client_hash_id", "client_id"],                           FACT_COLS, "client id"),
    "content":  resolve(["content_hash_id", "content_id"],                         FACT_COLS, "content id"),
    "imp":      resolve(["gsc_impressions", "impressions"],                        FACT_COLS, "impressions"),
    "clicks":   resolve(["gsc_clicks", "clicks"],                                  FACT_COLS, "clicks"),
    "pos":      resolve(["gsc_avg_position", "avg_position"],                      FACT_COLS, "avg position"),
    "gsc_flag": resolve(["gsc_data_available", "client_has_gsc", "has_gsc"],       FACT_COLS, "GSC availability flag"),
    "ga4_flag": resolve(["ga4_data_available", "client_has_ga4", "has_ga4"],       FACT_COLS, "GA4 availability flag"),
}

print(f"fact table exposes {len(FACT_COLS)} columns; the 8 this contract depends on resolve to:")
for k, v in COL.items():
    print(f"   {k:9} -> {v}")
print("\nAll fact columns:\n ", ", ".join(FACT_COLS))
print("\ndim_clients columns:\n ", ", ".join(CLIENT_COLS))

fact table exposes 31 columns; the 8 this contract depends on resolve to:
   date      -> report_date
   client    -> client_hash_id
   content   -> content_hash_id
   imp       -> gsc_impressions
   clicks    -> gsc_clicks
   pos       -> gsc_avg_position
   gsc_flag  -> gsc_data_available
   ga4_flag  -> ga4_data_available

All fact columns:
  report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month

dim_clients columns:
  client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date, client_updated_date, gsc_data_start, ga4_data_start


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1 · What one row means.**
One row = **one page, summarised over one calendar month**, i.e. one
`(client_hash_id, content_hash_id)` pair aggregated across all of its daily rows in
`month=2026-03`. The warehouse's own grain is finer — one row per *page per day per client*
(that is the row Haris showed being built by the nightly merge). My lane rolls those 31 page-days
up into one **page-month** row, because the decision I am supporting is made once per page, not
once per page per day: an editor picks *which pages* to review, not which page-days.

**2 · Which tables.**
`fact_content_daily_performance`, partition `month=2026-03` — everything in the contract is built
from it. `dim_clients` is read once, as **context only**, to check per-client history coverage
(§4). `dim_content` and `fact_content_query_90d` are deliberately *not* joined yet: the query
table's fixed 90-day window overlaps the months I will later use as an outcome window, so joining
it before I have aligned the windows is exactly the kind of quiet leak this assignment is about.

**3 · Which time window.**
Observation window = **2026-03-01 → 2026-03-31** (a mid-panel month; the `_sample` / June 2026
partition is left sealed). The decision moment is **2026-04-01**: an editor opens the review
queue on the 1st, holding only what was known on the 31st. Every feature must be computable from
`report_date <= 2026-03-31`. One feature deliberately looks *inside* the window (a last-14 vs
prior-14 split); none of them looks past it.

```text
        observation window (features + proxy)        outcome window (NOT used in ML-04)
   |--------------------------------------------|  |------------------------|      |----------|
   2026-03-01                          2026-03-31   2026-04-01     2026-04-30       2026-06 = SEALED
                                                |
                                       decision moment: the queue is ranked here
```

**4 · What I would rank (the proxy — not ground truth).**
A continuous **`ctr_gap_pp`**: how many percentage points below its *position-matched peers* a
page's observed click-through rate sits, over the window.

```text
ctr_pp_i     = 100 * clicks_i / impressions_i
peer_pp_i    = 100 * (SUM(clicks in tier) - clicks_i) / (SUM(impressions in tier) - impressions_i)
ctr_gap_pp_i = peer_pp_i - ctr_pp_i
```

This is a **rule I authored**, not an outcome anyone observed. It says "this page converts
visibility into clicks worse than comparable pages did", and nothing more — it does not say why,
and it does not say a rewrite would fix it. The queue the editor actually sees is ordered by
`expected_missed_clicks = impressions x ctr_gap_pp / 100`, where impressions is a **known
multiplier applied after the fact**, not something a model predicts.

Two upgrades over my ML-03 version, both fixing limitations I flagged there:
- the peer baseline is now **leave-one-out** (a page is never part of its own benchmark), which
  removes the circularity I flagged and could not test;
- the baseline is **volume-weighted** (pooled clicks ÷ pooled impressions) rather than a mean of
  per-page rates, so a 50-impression page no longer moves the benchmark as much as a
  200,000-impression one.

**5 · One thing I deliberately exclude.**
**All GA4 engagement columns** (sessions, engaged sessions, scroll, AI sessions) — for this
contract, not forever. Reason: the GA4 availability flag is *three-valued*. Rows before a client's
`ga4_data_start` are zero-**filled** with the flag FALSE, and millions more carry the flag NULL
with NULL metrics. Whether a page has usable GA4 is therefore a property of **the client**, not of
the page — so any GA4 feature I build partly encodes *which client this is*, and a client-holdout
split would then be measuring the wrong thing. Q3 below measures exactly how big that hole is
before I decide anything permanent.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The contract as constants, so the code below can never quietly disagree with the words above.
WINDOW_START, WINDOW_END = "2026-03-01", "2026-03-31"
DECISION_MOMENT          = "2026-04-01"     # the queue is ranked with data known up to WINDOW_END
SEALED_MONTHS            = ["2026-06"]      # the _sample partition: never used for label logic

# Eligibility rules for the lane slice (stated here, applied and counted in 3.4).
MIN_IMPRESSIONS = 500   # one click moves CTR by <= 0.2pp at this volume -> the gap is readable
MIN_ACTIVE_DAYS = 5     # need several days for within-window volatility/momentum to mean anything

CONTRACT = {
    "lane":              "Lane 4 - CTR / engagement opportunity scoring",
    "unit_of_analysis":  "one page-month: (client_hash_id, content_hash_id) aggregated over month=2026-03",
    "source_grain":      "report_date x client_hash_id x content_hash_id (page-day)",
    "tables_used":       ["fact_content_daily_performance (month=2026-03)", "dim_clients (context only)"],
    "observation_window": f"{WINDOW_START} .. {WINDOW_END}",
    "decision_moment":    DECISION_MOMENT,
    "target":            "ctr_gap_pp -- AUTHORED PROXY, leave-one-out volume-weighted tier baseline",
    "queue_ordering":    "expected_missed_clicks = impressions * ctr_gap_pp / 100",
    "deliberate_exclusion": "all GA4 engagement columns (three-valued availability flag == client fingerprint)",
    "sealed":            SEALED_MONTHS,
}

print("DATA CONTRACT (v1, ML-04)\n" + "-" * 70)
for k, v in CONTRACT.items():
    print(f"{k:22} : {v}")
print("-" * 70)
print(f"eligibility: impressions >= {MIN_IMPRESSIONS} in window AND active days >= {MIN_ACTIVE_DAYS} AND a valid position exists")

DATA CONTRACT (v1, ML-04)
----------------------------------------------------------------------
lane                   : Lane 4 - CTR / engagement opportunity scoring
unit_of_analysis       : one page-month: (client_hash_id, content_hash_id) aggregated over month=2026-03
source_grain           : report_date x client_hash_id x content_hash_id (page-day)
tables_used            : ['fact_content_daily_performance (month=2026-03)', 'dim_clients (context only)']
observation_window     : 2026-03-01 .. 2026-03-31
decision_moment        : 2026-04-01
target                 : ctr_gap_pp -- AUTHORED PROXY, leave-one-out volume-weighted tier baseline
queue_ordering         : expected_missed_clicks = impressions * ctr_gap_pp / 100
deliberate_exclusion   : all GA4 engagement columns (three-valued availability flag == client fingerprint)
sealed                 : ['2026-06']
----------------------------------------------------------------------
eligibility: impressions >= 500 in window AND active days

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every column I touch goes in **exactly one** bucket. The rule that decides the bucket is the
decision moment (2026-04-01): a column is a feature only if an editor could have known it on the
morning of the 1st, and only if knowing it does not hand back the answer.

| Bucket | Columns | Note |
|---|---|---|
| **Feature** (5, built from the window) | `log_impressions_31d`, `days_with_impressions_31d`, `position_volatility_31d`, `top_day_impression_share`, `momentum_log14v14` | all aggregated from `report_date <= 2026-03-31` |
| **Label / proxy — and its ingredients** | `ctr_gap_pp` · and therefore `clicks_31d`, `ctr_pp`, `peer_pp`, `tier_clicks`, `tier_impressions` | the proxy is *arithmetic* on these — none may ever be a feature |
| **Context** (group / join / split / read — never learned from) | `client_hash_id`, `content_hash_id`, `report_date`, `avg_position_31d`, `position_tier`, `impressions_31d`, `gsc_data_available`, `ga4_data_available`, `dim_clients.gsc_data_start` | IDs are pseudonyms; the rest is grouping, eligibility, and the known multiplier |
| **Excluded** (each with a why) | GA4 engagement columns · `fact_content_query_90d.*` · `dim_content.*` (for now) · any FlyRank product flag | see below |

**Why each exclusion:**

- **GA4 engagement columns** — the availability flag is three-valued (TRUE / FALSE / NULL), and
  availability is a property of the *client*. Any GA4 feature therefore partly encodes client
  identity, which is exactly what a client-holdout split is supposed to hold out. Excluded until
  I have measured the hole (Q3) and can handle it with explicit `has_` flags instead of a fill.
- **`fact_content_query_90d`** — fixed 90-day window that overlaps the months I intend to use as
  a forward outcome window later. Its `impressions_90d` / `*_last30` columns would contain my
  future label period. Only `*_prev30`-style columns could ever be safe, and only after I align
  the windows on paper first. Not joined in ML-04.
- **`dim_content`** — not excluded on principle (word count, intent and competition are genuine
  page-level features), but excluded *from this contract* because a join is a claim about key
  integrity, and I have not tested that key yet. Haris's least-glamorous slide is the reason: a
  bad join does not error, it silently returns nothing. It joins in ML-05, after a coverage check.
- **FlyRank product flags** (`health_score`, `needs_ctr_fix`, `is_quick_win`, …) — already dropped
  from the release, and rightly: they are FlyRank's *answers*. Predicting an answer from the
  inputs that built it is circular success.

**`avg_position_31d` is the interesting one, and it sits in Context on purpose.** Position is
perfectly knowable at the decision moment, so it is not leakage in the time sense. But position
*defines the peer group* whose baseline I subtract — hand it to a model as a feature and the model
can partly reconstruct `peer_pp`, i.e. reconstruct half of its own target. That is a soft,
structural leak rather than a temporal one. I keep position as the grouping key and audit whether
it can be promoted to a feature in ML-05, rather than quietly using it now.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# A contract that checks itself: buckets are declared, then tested for overlap.

FEATURES = {
    "log_impressions_31d":       "log1p of total GSC impressions in the window",
    "days_with_impressions_31d": "count of days in the window with >= 1 impression",
    "position_volatility_31d":   "stddev of daily avg position across active days",
    "top_day_impression_share":  "busiest day's impressions / window impressions",
    "momentum_log14v14":         "log ratio of last-14-day vs prior-14-day impressions, inside the window",
}

LABEL_AND_INGREDIENTS = {
    "ctr_gap_pp", "ctr_pp", "peer_pp", "clicks_31d", "tier_clicks", "tier_impressions",
    "expected_missed_clicks",
}

CONTEXT = {
    "client_hash_id", "content_hash_id", "report_date", "impressions_31d",
    "avg_position_31d", "position_tier", COL["gsc_flag"], COL["ga4_flag"], "gsc_data_start",
}

EXCLUDED = {
    "ga4_sessions / engaged_sessions / scroll / ai_sessions (any GA4 metric)":
        "availability flag is three-valued and is a CLIENT property -> encodes client identity",
    "fact_content_query_90d.impressions_90d / *_last30":
        "its fixed 90d window overlaps my future outcome window -> would contain the label period",
    "dim_content.* (this week only)":
        "a join is an untested claim about key integrity; bad joins fail silently, so ML-05 tests it first",
    "health_score / needs_ctr_fix / is_quick_win (FlyRank product flags)":
        "product ANSWERS, already dropped from the release; predicting them from their own inputs is circular",
}

# The one check that matters: no feature is also a label ingredient or a context/ID column.
assert set(FEATURES).isdisjoint(LABEL_AND_INGREDIENTS), "a label ingredient leaked into FEATURES"
assert set(FEATURES).isdisjoint(CONTEXT),               "a context column leaked into FEATURES"
assert "clicks_31d" not in FEATURES and "ctr_pp" not in FEATURES

print(f"{len(FEATURES)} features / {len(LABEL_AND_INGREDIENTS)} label-side columns / "
      f"{len(CONTEXT)} context columns / {len(EXCLUDED)} exclusion rules")
print("bucket disjointness check: PASS -- no feature is a label ingredient or a context column\n")
for k, why in EXCLUDED.items():
    print(f"EXCLUDED  {k}\n          why: {why}")

5 features / 7 label-side columns / 9 context columns / 4 exclusion rules
bucket disjointness check: PASS -- no feature is a label ingredient or a context column

EXCLUDED  ga4_sessions / engaged_sessions / scroll / ai_sessions (any GA4 metric)
          why: availability flag is three-valued and is a CLIENT property -> encodes client identity
EXCLUDED  fact_content_query_90d.impressions_90d / *_last30
          why: its fixed 90d window overlaps my future outcome window -> would contain the label period
EXCLUDED  dim_content.* (this week only)
          why: a join is an untested claim about key integrity; bad joins fail silently, so ML-05 tests it first
EXCLUDED  health_score / needs_ctr_fix / is_quick_win (FlyRank product flags)
          why: product ANSWERS, already dropped from the release; predicting them from their own inputs is circular


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims, three queries, on the mid-panel partition `month=2026-03`:

| # | The claim from §1 | The query that could disprove it |
|---|---|---|
| **Q1** | the source grain really is one row per page per day per client | group by those three columns and look for any group with more than one row |
| **Q2** | my slice is one calendar month, 2026-03-01 → 2026-03-31 | row count, distinct pages/clients, and `MIN`/`MAX(report_date)` |
| **Q3** | availability is three-valued and must be filtered with `IS TRUE` | count TRUE / FALSE / NULL, and show what `= FALSE` silently drops |

Each cell prints a **verdict sentence with the number in it**, so the claim and its evidence stay
attached to each other.

### Q1 — Grain: is one row really one page × one day × one client?

If the grain holds, grouping by those three columns can never produce a group with more than one
row. Zero rows back = the grain holds. This is the heaviest of the three checks (it reads three
columns across the whole month) — expect a minute or two.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Q1: grain probe --------------------------------------------------------
q1 = con.sql(f"""
    SELECT {COL['date']} AS report_date,
           {COL['client']} AS client_hash_id,
           {COL['content']} AS content_hash_id,
           COUNT(*) AS n
    FROM {T['fact_month']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"groups with more than one row (showing up to 5): {len(q1)}")
if len(q1) == 0:
    print(f"VERDICT: grain holds. One row = one {COL['content']} x one {COL['date']} x one "
          f"{COL['client']}. My page-month row is therefore a roll-up of up to 31 of these.")
else:
    print("VERDICT: grain does NOT hold -- the contract sentence in section 1 must be rewritten.")
    display(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

groups with more than one row (showing up to 5): 0
VERDICT: grain holds. One row = one content_hash_id x one report_date x one client_hash_id. My page-month row is therefore a roll-up of up to 31 of these.


### Q2 — Counts and window: how big is the slice, and does it really cover one month?

The contract claims a calendar month. A partition folder is a *promise* about dates, not a proof —
this checks the dates themselves, and gives the numbers anyone should be able to predict from
reading §1.

In [11]:
# --- Q2: row count, distinct pages/clients, date span -----------------------
q2 = con.sql(f"""
    SELECT COUNT(*)                                        AS page_day_rows,
           COUNT(DISTINCT {COL['content']})                AS distinct_pages,
           COUNT(DISTINCT {COL['client']})                 AS distinct_clients,
           MIN({COL['date']})                              AS first_date,
           MAX({COL['date']})                              AS last_date,
           COUNT(DISTINCT {COL['date']})                   AS distinct_days
    FROM {T['fact_month']}
""").df().iloc[0]

span_ok  = str(q2.first_date)[:10] == WINDOW_START and str(q2.last_date)[:10] == WINDOW_END
print(f"page-day rows in month={MONTH} : {q2.page_day_rows:,}")
print(f"distinct pages                 : {q2.distinct_pages:,}")
print(f"distinct clients               : {q2.distinct_clients:,}")
print(f"date span                      : {str(q2.first_date)[:10]} -> {str(q2.last_date)[:10]} "
      f"({q2.distinct_days} distinct days)")
print(f"avg page-days per page         : {q2.page_day_rows / q2.distinct_pages:.1f} of {q2.distinct_days}")
print()
print(f"VERDICT: the partition's dates {'match' if span_ok else 'DO NOT match'} the window claimed in "
      f"section 1 ({WINDOW_START} .. {WINDOW_END}). Rolling {q2.page_day_rows:,} page-days up to the "
      f"page-month grain gives at most {q2.distinct_pages:,} rows -- before eligibility filtering.")

page-day rows in month=2026-03 : 9,841,378
distinct pages                 : 331,437
distinct clients               : 55
date span                      : 2026-03-01 -> 2026-03-31 (31 distinct days)
avg page-days per page         : 29.7 of 31

VERDICT: the partition's dates match the window claimed in section 1 (2026-03-01 .. 2026-03-31). Rolling 9,841,378 page-days up to the page-month grain gives at most 331,437 rows -- before eligibility filtering.


### Q3 — Availability: the two kinds of nothing

A `0` in an impressions column means *we looked and there was nothing*. A `NULL` means *we could
not look*. The flags that tell them apart are three-valued, so `= TRUE` is not a filter — it is a
bug that returns fewer rows than you think and never says so.

This query counts TRUE / FALSE / NULL for both flags, and prints the number that matters: how many
rows `= FALSE` silently loses compared with `IS NOT TRUE`.

In [12]:
# --- Q3: availability funnel, filtered with IS TRUE -------------------------
q3 = con.sql(f"""
    SELECT COUNT(*)                                                          AS all_rows,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS TRUE)                 AS gsc_true,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS FALSE)                AS gsc_false,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS NULL)                 AS gsc_null,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS TRUE)                 AS ga4_true,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS FALSE)                AS ga4_false,
           COUNT(*) FILTER (WHERE {COL['ga4_flag']} IS NULL)                 AS ga4_null,
           COUNT(*) FILTER (WHERE {COL['gsc_flag']} IS TRUE
                              AND {COL['imp']} > 0)                          AS gsc_true_with_impressions
    FROM {T['fact_month']}
""").df().iloc[0]

n = q3.all_rows
pct = lambda x: f"{x:>12,}  ({x / n:6.2%})"
print(f"rows in month={MONTH}: {n:,}\n")
print(f"{COL['gsc_flag']:<22} IS TRUE  {pct(q3.gsc_true)}")
print(f"{'':<22} IS FALSE {pct(q3.gsc_false)}")
print(f"{'':<22} IS NULL  {pct(q3.gsc_null)}   <- 'we could not look'")
print()
print(f"{COL['ga4_flag']:<22} IS TRUE  {pct(q3.ga4_true)}")
print(f"{'':<22} IS FALSE {pct(q3.ga4_false)}")
print(f"{'':<22} IS NULL  {pct(q3.ga4_null)}   <- 'we could not look'")
print()
print(f"GSC IS TRUE *and* impressions > 0 : {q3.gsc_true_with_impressions:,} rows "
      f"({q3.gsc_true_with_impressions / n:.2%}) -- the rows my lane can actually see")
print()
print("The trap, in numbers:")
print(f"  rows caught by  {COL['ga4_flag']} = FALSE      : {q3.ga4_false:,}")
print(f"  rows caught by  {COL['ga4_flag']} IS NOT TRUE  : {q3.ga4_false + q3.ga4_null:,}")
print(f"  difference (the NULLs a '= FALSE' filter drops silently): {q3.ga4_null:,}")
print()
print("VERDICT: availability is three-valued, so every filter in this notebook uses IS TRUE / "
      "IS NOT TRUE. This is also the measured reason GA4 columns are excluded in section 1: the "
      "hole is large and it is a client property, not a page property.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in month=2026-03: 9,841,378

gsc_data_available     IS TRUE     3,611,061  (36.69%)
                       IS FALSE    6,230,317  (63.31%)
                       IS NULL             0  ( 0.00%)   <- 'we could not look'

ga4_data_available     IS TRUE       413,966  ( 4.21%)
                       IS FALSE    6,408,671  (65.12%)
                       IS NULL     3,018,741  (30.67%)   <- 'we could not look'

GSC IS TRUE *and* impressions > 0 : 3,611,061 rows (36.69%) -- the rows my lane can actually see

The trap, in numbers:
  rows caught by  ga4_data_available = FALSE      : 6,408,671
  rows caught by  ga4_data_available IS NOT TRUE  : 9,427,412
  difference (the NULLs a '= FALSE' filter drops silently): 3,018,741

VERDICT: availability is three-valued, so every filter in this notebook uses IS TRUE / IS NOT TRUE. This is also the measured reason GA4 columns are excluded in section 1: the hole is large and it is a client property, not a page property.


### 3.4 The lane slice and the five features

One query, aggregating page-days into page-months. Everything it computes is bounded by
`report_date <= 2026-03-31`, so everything survives the decision-moment test. The heavy work stays
in SQL and only the small aggregate comes back to pandas — the whole point of the DuckDB pattern.

Two grain guards are worth naming: the impression-weighted position uses
`SUM(position × impressions) / SUM(impressions)` rather than a plain average of daily averages
(a day with 3 impressions should not count as much as a day with 3,000), and it is computed only
over days that actually had impressions and a real position — `avg_position = 0` means *no data*,
not rank zero.

In [13]:
# --- The lane slice: page-days -> page-months, all inside the window --------
# Momentum splits the window into two equal 14-day halves: Mar 18-31 vs Mar 04-17.
# (Mar 01-03 is deliberately unused so the two halves are the same length.)
LAST14_FROM, PREV14_FROM = "2026-03-18", "2026-03-04"

lane = con.sql(f"""
    WITH page_month AS (
        SELECT
            {COL['client']}   AS client_hash_id,
            {COL['content']}  AS content_hash_id,

            SUM({COL['imp']})                                                    AS impressions_31d,
            SUM({COL['clicks']})                                                 AS clicks_31d,
            COUNT(DISTINCT CASE WHEN {COL['imp']} > 0 THEN {COL['date']} END)     AS days_with_impressions_31d,
            MAX({COL['imp']})                                                    AS top_day_impressions,

            SUM({COL['pos']} * {COL['imp']}) FILTER (
                WHERE {COL['imp']} > 0 AND {COL['pos']} IS NOT NULL AND {COL['pos']} > 0)
                                                                                 AS pos_weight_num,
            SUM({COL['imp']}) FILTER (
                WHERE {COL['imp']} > 0 AND {COL['pos']} IS NOT NULL AND {COL['pos']} > 0)
                                                                                 AS pos_weight_den,
            STDDEV_SAMP({COL['pos']}) FILTER (
                WHERE {COL['imp']} > 0 AND {COL['pos']} IS NOT NULL AND {COL['pos']} > 0)
                                                                                 AS position_volatility_31d,

            SUM({COL['imp']}) FILTER (WHERE {COL['date']} >= DATE '{LAST14_FROM}') AS imp_last14,
            SUM({COL['imp']}) FILTER (WHERE {COL['date']} >= DATE '{PREV14_FROM}'
                                        AND {COL['date']} <  DATE '{LAST14_FROM}') AS imp_prev14
        FROM {T['fact_month']}
        WHERE {COL['gsc_flag']} IS TRUE          -- IS TRUE, never = TRUE
        GROUP BY 1, 2
    )
    SELECT *,
           pos_weight_num / NULLIF(pos_weight_den, 0) AS avg_position_31d
    FROM page_month
    WHERE impressions_31d >= {MIN_IMPRESSIONS}
      AND days_with_impressions_31d >= {MIN_ACTIVE_DAYS}
      AND pos_weight_den > 0
""").df()

print(f"lane slice: {len(lane):,} page-month rows, {lane.client_hash_id.nunique()} clients")
lane[["impressions_31d", "clicks_31d", "days_with_impressions_31d", "avg_position_31d"]].describe().round(2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lane slice: 61,881 page-month rows, 36 clients


,impressions_31d,clicks_31d,days_with_impressions_31d,avg_position_31d
count,61881.00,61881.00,61881.00,61881.00
mean,4345.17,12.81,29.53,11.45
std,8516.19,44.00,3.83,11.29
min,500.00,0.00,5.00,0.06
25%,941.00,1.00,31.00,4.14
50%,1900.00,4.00,31.00,6.79
75%,4548.00,12.00,31.00,14.87
max,617124.00,5668.00,31.00,87.71


In [14]:
# --- The eligibility funnel, stated honestly --------------------------------
funnel = con.sql(f"""
    SELECT
      COUNT(*)                                                                   AS pages_with_gsc_true,
      COUNT(*) FILTER (WHERE impressions_31d >= {MIN_IMPRESSIONS})               AS after_volume_floor,
      COUNT(*) FILTER (WHERE impressions_31d >= {MIN_IMPRESSIONS}
                         AND days_with_impressions_31d >= {MIN_ACTIVE_DAYS})     AS after_active_days
    FROM (
      SELECT {COL['client']} AS c, {COL['content']} AS p,
             SUM({COL['imp']}) AS impressions_31d,
             COUNT(DISTINCT CASE WHEN {COL['imp']} > 0 THEN {COL['date']} END) AS days_with_impressions_31d
      FROM {T['fact_month']}
      WHERE {COL['gsc_flag']} IS TRUE
      GROUP BY 1, 2
    )
""").df().iloc[0]

print("Eligibility funnel (pages, month=2026-03)")
print(f"  GSC availability IS TRUE                   : {funnel.pages_with_gsc_true:,}")
print(f"  ... and >= {MIN_IMPRESSIONS} impressions in the window : {funnel.after_volume_floor:,} "
      f"({funnel.after_volume_floor / funnel.pages_with_gsc_true:.1%})")
print(f"  ... and >= {MIN_ACTIVE_DAYS} active days                     : {funnel.after_active_days:,} "
      f"({funnel.after_active_days / funnel.pages_with_gsc_true:.1%})")
print(f"  ... and a valid position exists            : {len(lane):,} "
      f"({len(lane) / funnel.pages_with_gsc_true:.1%})  <- the lane slice")
print()
print(f"OBSERVED: the volume floor is what removes most pages. That is a deliberate, stated "
      f"restriction of scope -- my lane speaks only about pages already visible enough for a CTR "
      f"gap to be readable, and says nothing about the {funnel.pages_with_gsc_true - len(lane):,} "
      f"pages below the floor.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligibility funnel (pages, month=2026-03)
  GSC availability IS TRUE                   : 176,738
  ... and >= 500 impressions in the window : 61,924 (35.0%)
  ... and >= 5 active days                     : 61,881 (35.0%)
  ... and a valid position exists            : 61,881 (35.0%)  <- the lane slice

OBSERVED: the volume floor is what removes most pages. That is a deliberate, stated restriction of scope -- my lane speaks only about pages already visible enough for a CTR gap to be readable, and says nothing about the 114,857 pages below the floor.


In [15]:
# --- Build the five features + the proxy -----------------------------------
# Tier thresholds are the documented, transparent ones (<=3 / <=10 / <=20 / <=50 / >50).
def position_tier(p):
    if p <= 3:  return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

lane["position_tier"] = lane["avg_position_31d"].apply(position_tier)          # CONTEXT (grouping key)

# --- the five features (all knowable at 2026-04-01) ---
lane["log_impressions_31d"]      = np.log1p(lane["impressions_31d"])
lane["top_day_impression_share"] = lane["top_day_impressions"] / lane["impressions_31d"]
# fillna(0) is legitimate here and only here: the page's GSC flag IS TRUE and the days exist in
# the window, so an empty half means "we looked and there was nothing" -- the 0 kind of nothing.
lane["momentum_log14v14"]        = np.log((lane["imp_last14"].fillna(0) + 1) /
                                          (lane["imp_prev14"].fillna(0) + 1))
# days_with_impressions_31d and position_volatility_31d come straight from SQL.

# --- the proxy: leave-one-out, volume-weighted tier baseline ---
tier_clicks = lane.groupby("position_tier")["clicks_31d"].transform("sum")
tier_imps   = lane.groupby("position_tier")["impressions_31d"].transform("sum")

lane["ctr_pp"]     = 100 * lane["clicks_31d"] / lane["impressions_31d"]
lane["peer_pp"]    = 100 * (tier_clicks - lane["clicks_31d"]) / (tier_imps - lane["impressions_31d"])
lane["peer_pp"]    = lane["peer_pp"].replace([np.inf, -np.inf], np.nan)        # a tier of one page
lane["ctr_gap_pp"] = lane["peer_pp"] - lane["ctr_pp"]                          # THE PROXY
lane["expected_missed_clicks"] = lane["impressions_31d"] * lane["ctr_gap_pp"] / 100

before = len(lane)
lane = lane.dropna(subset=list(FEATURES) + ["ctr_gap_pp"]).copy()
print(f"rows dropped for a missing feature or proxy value: {before - len(lane):,} "
      f"(never filled -- a NULL means 'could not look')")
print(f"modelling frame: {len(lane):,} pages, {lane.client_hash_id.nunique()} clients\n")

print("Peer baseline per tier (leave-one-out is per row; pooled values shown for readability):")
tiers = (lane.groupby("position_tier")
             .agg(pages=("ctr_gap_pp", "size"),
                  pooled_ctr_pp=("ctr_pp", lambda s: np.average(s, weights=lane.loc[s.index, "impressions_31d"])),
                  median_gap_pp=("ctr_gap_pp", "median"),
                  underperforming=("ctr_gap_pp", lambda s: (s > 0).mean()))
             .round(3))
display(tiers)

rows dropped for a missing feature or proxy value: 0 (never filled -- a NULL means 'could not look')
modelling frame: 61,881 pages, 36 clients

Peer baseline per tier (leave-one-out is per row; pooled values shown for readability):


,pages,pooled_ctr_pp,median_gap_pp,underperforming
position_tier,,,,
deep,684,0.034,0.034,0.768
page_1,32300,0.324,0.110,0.646
page_3_5,10469,0.140,0.058,0.653
striking,10550,0.324,0.158,0.712
top_3,7878,0.388,0.152,0.684


In [16]:
# --- Did the two ML-03 limitations actually close? --------------------------
# ML-03 flagged: (a) zero-CTR pages all tie at the maximum score, regardless of volume;
#                (b) the tier baseline included the page being scored (circularity).
v1_ties = (lane["ctr_gap_pp"] == lane["ctr_gap_pp"].max()).sum()
v2_ties = (lane["expected_missed_clicks"] == lane["expected_missed_clicks"].max()).sum()

zero_ctr = lane[lane["clicks_31d"] == 0]
print(f"pages with zero clicks in the window: {len(zero_ctr):,} ({len(zero_ctr)/len(lane):.1%})")
print(f"  their impressions range: {zero_ctr['impressions_31d'].min():,.0f} .. "
      f"{zero_ctr['impressions_31d'].max():,.0f}")
print()
print(f"(a) pages tied at the maximum RATE gap (ML-03's score)      : {v1_ties:,}")
print(f"    pages tied at the maximum MISSED-CLICKS score (v2 queue) : {v2_ties:,}")
print(f"    -> the volume multiplier separates pages the rate gap could not tell apart.")
print()
# (b) how much did self-inclusion distort the old baseline?
naive_peer = lane.groupby("position_tier")["ctr_pp"].transform("mean")      # ML-03 style: self-included, mean of rates
shift = (naive_peer - lane["peer_pp"]).abs()
print(f"(b) |ML-03 baseline - leave-one-out baseline|, in percentage points:")
print(f"    median {shift.median():.3f}pp | mean {shift.mean():.3f}pp | max {shift.max():.3f}pp")
print(f"    -> OBSERVED, directional: this measures the circularity I could only flag in ML-03. "
      f"Both changes (leave-one-out and volume weighting) move the baseline together, so this is "
      f"the combined size of the correction, not proof that either one alone mattered.")

pages with zero clicks in the window: 10,783 (17.4%)
  their impressions range: 500 .. 44,707

(a) pages tied at the maximum RATE gap (ML-03's score)      : 1
    pages tied at the maximum MISSED-CLICKS score (v2 queue) : 1
    -> the volume multiplier separates pages the rate gap could not tell apart.

(b) |ML-03 baseline - leave-one-out baseline|, in percentage points:
    median 0.005pp | mean 0.018pp | max 0.057pp
    -> OBSERVED, directional: this measures the circularity I could only flag in ML-03. Both changes (leave-one-out and volume weighting) move the baseline together, so this is the combined size of the correction, not proof that either one alone mattered.


### 3.5 The five features — "knowable at the decision moment because…"

The decision moment is **2026-04-01**. Each line below is the reason the feature passes that test.

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `log_impressions_31d` | it sums GSC impressions with `report_date <= 2026-03-31`; the last day it can see is the day before the queue is built. `log1p` because search traffic is heavy-tailed, and the transform uses no information the raw column didn't already have. |
| 2 | `days_with_impressions_31d` | it counts distinct days inside the window that had at least one impression — a coverage measure of the *past*. It was the strongest feature in the starter pipeline, so it is a signal worth carrying, not a filler. |
| 3 | `position_volatility_31d` | the standard deviation of daily positions already recorded in the window. Nothing after the 31st enters it. Days with no impressions or `position = 0` ("no data") are excluded rather than treated as rank zero. |
| 4 | `top_day_impression_share` | the busiest day's share of window impressions — computed from the same 31 days. It separates a page with one spike from a page with steady demand, two situations whose CTR should not be read the same way. |
| 5 | `momentum_log14v14` | both halves (Mar 18–31 and Mar 04–17) sit **inside** the observation window. It is momentum measured in the past, not a peek at April. `log((a+1)/(b+1))` keeps it defined when a half is zero without inventing a value. |

**What is *not* here, and why that is the point.** `clicks_31d` is equally knowable on 2026-04-01
— and it is disqualified anyway, because the proxy is arithmetic on it. Knowability is necessary,
not sufficient. Section 3.6 shows what happens when I forget that.

In [17]:
# The features and their availability reason, printed next to the frame they describe.
print(f"decision moment: {DECISION_MOMENT} -- every feature below is computed from "
      f"report_date <= {WINDOW_END}\n")
for i, (name, what) in enumerate(FEATURES.items(), 1):
    print(f"{i}. {name:28} {what}")
print()
print(lane[list(FEATURES)].describe().round(3))

decision moment: 2026-04-01 -- every feature below is computed from report_date <= 2026-03-31

1. log_impressions_31d          log1p of total GSC impressions in the window
2. days_with_impressions_31d    count of days in the window with >= 1 impression
3. position_volatility_31d      stddev of daily avg position across active days
4. top_day_impression_share     busiest day's impressions / window impressions
5. momentum_log14v14            log ratio of last-14-day vs prior-14-day impressions, inside the window

       log_impressions_31d  days_with_impressions_31d  position_volatility_31d  top_day_impression_share  momentum_log14v14
count            61881.000                  61881.000                61881.000                 61881.000          61881.000
mean                 7.710                     29.533                    4.485                     0.083              0.234
std                  1.046                      3.831                    4.003                     0.053       

### 3.6 The trap: one deliberate leak, then removed

Notebook 02 showed leakage on the starter CSV by adding `trend_pct`. Here is the same lesson on
real warehouse data, with a column that looks far more innocent.

**The leak: `clicks_31d`.** It passes every test a beginner applies. It is a real observed metric,
it is knowable at the decision moment, and "how many clicks did this page get" sounds like an
obvious feature about the page. But the proxy is

```text
ctr_gap_pp = peer_pp - 100 * clicks_31d / impressions_31d
```

and `impressions_31d` is already a feature (as `log_impressions_31d`), while `peer_pp` only takes
five distinct-ish values because it is a tier baseline. Hand the model `clicks_31d` and it is no
longer predicting anything — it is doing arithmetic it has already been given both operands for.

The score is a **grouped** split: whole clients held out, mirroring the starter pipeline's
`client_holdout` strategy, so the number is not inflated by a page's clientmates sitting in train.
Spearman is the headline metric because this lane is a **ranking** problem — what matters is
whether the order is right, not whether the value is.

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

y      = lane["ctr_gap_pp"].values
groups = lane["client_hash_id"].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(splitter.split(lane, y, groups))
print(f"grouped split: {len(tr):,} train rows / {len(te):,} test rows | "
      f"{lane.iloc[tr].client_hash_id.nunique()} train clients / "
      f"{lane.iloc[te].client_hash_id.nunique()} held-out clients (no client appears in both)\n")

def quick_score(cols, label):
    X = lane[cols].values
    m = RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1).fit(X[tr], y[tr])
    p = m.predict(X[te])
    rho = spearmanr(p, y[te])[0]
    r2  = r2_score(y[te], p)
    print(f"{label:<34} Spearman {rho:6.3f} | R2 {r2:7.3f}")
    return rho, r2

honest_cols = list(FEATURES)
leaky_cols  = honest_cols + ["clicks_31d"]          # <-- the deliberate leak

print("held-out-client scores")
print("-" * 62)
rho_h, r2_h = quick_score(honest_cols, "5 honest features")
rho_l, r2_l = quick_score(leaky_cols,  "+ clicks_31d  (DELIBERATE LEAK)")
print("-" * 62)
print(f"the leak buys {rho_l - rho_h:+.3f} Spearman and {r2_l - r2_h:+.3f} R2 -- "
      f"and every point of it is arithmetic, not signal.")

grouped split: 33,676 train rows / 28,205 test rows | 27 train clients / 9 held-out clients (no client appears in both)

held-out-client scores
--------------------------------------------------------------
5 honest features                  Spearman  0.156 | R2  -0.169
+ clicks_31d  (DELIBERATE LEAK)    Spearman  0.896 | R2   0.941
--------------------------------------------------------------
the leak buys +0.740 Spearman and +1.110 R2 -- and every point of it is arithmetic, not signal.


In [19]:
# --- Remove the leak and keep the honest number ----------------------------
leaky_cols.remove("clicks_31d")
assert "clicks_31d" not in leaky_cols
assert set(FEATURES).isdisjoint(LABEL_AND_INGREDIENTS)   # the section-2 guard, re-run after the demo

HONEST_BASELINE = {"features": honest_cols, "spearman": round(float(rho_h), 3),
                   "r2": round(float(r2_h), 3), "validation": "GroupShuffleSplit on client_hash_id",
                   "seed": 42}

print("leak removed. The number this contract carries forward is the honest one:")
print(f"   Spearman {HONEST_BASELINE['spearman']:.3f} | R2 {HONEST_BASELINE['r2']:.3f} "
      f"on held-out clients, 5 features, seed 42")
print()
print("Reading it honestly: this is a floor, not a result. It measures how far five "
      "volume-and-stability features can order an AUTHORED proxy on clients the model never saw. "
      "If that number is small, that is itself information -- it says the CTR gap is mostly NOT "
      "explained by how much traffic a page gets or how steadily it gets it, which is exactly the "
      "argument for adding page meaning (dim_content) and query mix in ML-05. The near-perfect "
      "number above is the failure mode, not the target.")

leak removed. The number this contract carries forward is the honest one:
   Spearman 0.156 | R2 -0.169 on held-out clients, 5 features, seed 42

Reading it honestly: this is a floor, not a result. It measures how far five volume-and-stability features can order an AUTHORED proxy on clients the model never saw. If that number is small, that is itself information -- it says the CTR gap is mostly NOT explained by how much traffic a page gets or how steadily it gets it, which is exactly the argument for adding page meaning (dim_content) and query mix in ML-05. The near-perfect number above is the failure mode, not the target.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### The named limitation: my slice is a left-censored, unbalanced panel

**The claim:** a single calendar-month window is not the same window for every client. Exports
have no time machine — they begin the day someone switched them on, and `dim_clients.gsc_data_start`
records that day per client. Any client whose history starts *inside or after* March 2026
contributes a partial month, and their pages then look low-volume and low-coverage for a
**measurement** reason, not a performance reason. Because my eligibility rules are volume-based,
those clients are quietly filtered out — and the queue silently becomes a queue about
well-instrumented, long-tenured clients. The cell below measures how much of my slice that is.

### The other four, stated plainly

- **This data can never tell me *why* a CTR gap exists.** Titles, meta descriptions, URLs and raw
  queries were dropped from the release for privacy, and SERP layout (ads, AI overviews, featured
  snippets) was never in it. So a flagged page is *"observed to convert visibility into clicks
  worse than position-matched peers"* — never *"has a weak title"*. The output is decision-support
  for where an editor looks first; the diagnosis stays human.
- **There is no counterfactual.** I never observe the same page both reviewed and not reviewed, so
  nothing here can support a claim that fixing a flagged page *will* close its gap. That needs a
  designed experiment and is out of scope for the capstone.
- **Recent days are still moving.** The nightly job re-merges a rolling five-day window because
  exports arrive late and get revised, so the newest days are the least settled. A second reason
  to develop on a mid-panel month rather than the edge of the panel.
  - **Window overlap is a live trap for later.** `fact_content_query_90d` covers a fixed 90-day
  window that overlaps the months I will eventually use as an outcome window. When I add
  query-mix features in ML-05, the window alignment has to be drawn before the join is written —
  not after the score looks good.

### And one honest limit of the proxy itself

`ctr_gap_pp` is a rule I wrote, so "underperforming" is defined by my tier thresholds and my
volume floor, not by anything the world confirmed. A page can sit at the top of my queue and be
performing exactly as it should — a navigational page, or one whose peers are a bad comparison
set. Until a forward-looking outcome exists (*did CTR improve in the window after a page was
flagged?*), every number in this notebook describes an **authored ranking**, and true Precision@K
is not yet available to me.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Measure the named limitation: unbalanced, left-censored panel ---------
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {T['dim_clients']}
""").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"], errors="coerce")

win_start = pd.Timestamp(WINDOW_START)
full_hist = clients["gsc_data_start"] <= win_start
print(f"clients in dim_clients                                   : {len(clients):,}")
print(f"  with GSC history starting on/before {WINDOW_START}       : {int(full_hist.sum())}")
print(f"  starting DURING or AFTER the window (partial month)     : {int((clients['gsc_data_start'] > win_start).sum())}")
print(f"  with NO gsc_data_start recorded at all (NULL)           : {int(clients['gsc_data_start'].isna().sum())}")
print()

in_lane = set(lane["client_hash_id"].unique())
lane_clients = clients[clients["client_hash_id"].isin(in_lane)]
print(f"clients that survive into my lane slice                   : {len(lane_clients)} of {len(clients)} "
      f"({len(lane_clients)/len(clients):.1%})")
if len(lane_clients):
    print(f"  their earliest / latest gsc_data_start                 : "
          f"{str(lane_clients['gsc_data_start'].min())[:10]} .. {str(lane_clients['gsc_data_start'].max())[:10]}")

share = lane.groupby("client_hash_id").size().sort_values(ascending=False) / len(lane)
print()
print(f"client concentration inside the lane slice: largest client = {share.iloc[0]:.1%} of rows, "
      f"top 3 = {share.head(3).sum():.1%}")
print()
print(f"VERDICT (observed, directional): my slice describes {len(lane_clients)} well-instrumented "
      f"clients, not the client base. Client-holdout validation is therefore mandatory, and any "
      f"result must be read as 'held on {len(lane_clients)} clients with usable March history' -- "
      f"never as a statement about clients with short or missing history.")

clients in dim_clients                                   : 104
  with GSC history starting on/before 2026-03-01       : 52
  starting DURING or AFTER the window (partial month)     : 15
  with NO gsc_data_start recorded at all (NULL)           : 37

clients that survive into my lane slice                   : 36 of 104 (34.6%)
  their earliest / latest gsc_data_start                 : 2025-01-27 .. 2026-03-27

client concentration inside the lane slice: largest client = 23.0% of rows, top 3 = 57.1%

VERDICT (observed, directional): my slice describes 36 well-instrumented clients, not the client base. Client-holdout validation is therefore mandatory, and any result must be read as 'held on 36 clients with usable March history' -- never as a statement about clients with short or missing history.


In [21]:
# --- Receipts: the contract, auto-filled with the numbers that were measured -
import json, os

receipts = {
    "assignment": "ML-04 - Search Intelligence Data Contract",
    "lane": CONTRACT["lane"],
    "development_partition": f"month={MONTH}",
    "sealed_partitions": SEALED_MONTHS,
    "contract": CONTRACT,
    "verified": {
        "Q1_grain_violations": int(len(q1)),
        "Q2_page_day_rows": int(q2.page_day_rows),
        "Q2_distinct_pages": int(q2.distinct_pages),
        "Q2_distinct_clients": int(q2.distinct_clients),
        "Q2_date_span": [str(q2.first_date)[:10], str(q2.last_date)[:10]],
        "Q3_gsc_true_rows": int(q3.gsc_true),
        "Q3_ga4_true_rows": int(q3.ga4_true),
        "Q3_ga4_null_rows_missed_by_equals_false": int(q3.ga4_null),
    },
    "lane_slice": {
        "rows": int(len(lane)),
        "clients": int(lane["client_hash_id"].nunique()),
        "eligibility": {"min_impressions": MIN_IMPRESSIONS, "min_active_days": MIN_ACTIVE_DAYS},
    },
    "features": list(FEATURES),
    "proxy": "ctr_gap_pp (leave-one-out, volume-weighted tier baseline) - AUTHORED, not ground truth",
    "leak_experiment": {
        "leaked_column": "clicks_31d",
        "leaked_spearman": round(float(rho_l), 3),
        "honest_spearman": round(float(rho_h), 3),
        "status": "removed",
    },
    "honest_baseline": HONEST_BASELINE,
}

print(json.dumps(receipts, indent=2))

# Committed receipts live in work/outputs/ (JSONs are kept in git; datasets are not).
try:
    os.makedirs("work/outputs", exist_ok=True)
    with open("work/outputs/ml04_data_contract_receipts.json", "w") as f:
        json.dump(receipts, f, indent=2)
    print("\nsaved -> work/outputs/ml04_data_contract_receipts.json")
except Exception as e:
    print(f"\n(receipts not written to disk in this environment: {e})")

{
  "assignment": "ML-04 - Search Intelligence Data Contract",
  "lane": "Lane 4 - CTR / engagement opportunity scoring",
  "development_partition": "month=2026-03",
  "sealed_partitions": [
    "2026-06"
  ],
  "contract": {
    "lane": "Lane 4 - CTR / engagement opportunity scoring",
    "unit_of_analysis": "one page-month: (client_hash_id, content_hash_id) aggregated over month=2026-03",
    "source_grain": "report_date x client_hash_id x content_hash_id (page-day)",
    "tables_used": [
      "fact_content_daily_performance (month=2026-03)",
      "dim_clients (context only)"
    ],
    "observation_window": "2026-03-01 .. 2026-03-31",
    "decision_moment": "2026-04-01",
    "target": "ctr_gap_pp -- AUTHORED PROXY, leave-one-out volume-weighted tier baseline",
    "queue_ordering": "expected_missed_clicks = impressions * ctr_gap_pp / 100",
    "deliberate_exclusion": "all GA4 engagement columns (three-valued availability flag == client fingerprint)",
    "sealed": [
      "2026-06

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.